In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sqlalchemy import create_engine
from urllib.parse import quote_plus


This notebook is used to apply the corrections for the old models.

In [ ]:
def construct_query():
    """
    Constructs an SQL query to fetch the required data based on the target group.
    """
    return f"""
        SELECT b.flight_id,
                b.target_group,
                f.start_date,
                f.end_date,
                b.ster_grp,
                b.[ster_1+]
        FROM [dbo].[tv.bereik_ster] b
        LEFT JOIN [dbo].[tv.flights_ster] f ON b.flight_id = f.flight_id
        WHERE b.ster_grp > 0
        AND b.campaign_duration >= 7
        AND b.daypart = 'total'
        ORDER BY start_date;
    """

def create_sql_alchemy_engine():
    """
    Creates and returns a SQLAlchemy engine for connecting to a SQL Server database.
    """
    quoted = quote_plus(
        "Driver={ODBC+Driver+17+for+SQL+Server};"
        "Server=tcp:stercampsql.database.windows.net;"
        "Database=SterCampNew;"
        "uid=stercampadmin;"
        f"pwd={os.getenv('SQLDB')}"
    )
    return create_engine(f"mssql+pyodbc:///?odbc_connect={quoted}")

def query_to_df(query: str):
    """
    Executes the SQL query and returns the result as a DataFrame.
    """
    engine = create_sql_alchemy_engine()
    return pd.read_sql_query(query, engine)

# Function to create rounded buckets within each target group
def calculate_bucket_bounds(data, n_buckets=4, rounding_base=10):
    bucket_bounds = {}
    for tg, group in data.groupby('target_group'):
        # Calculate quantiles for ster_grp to divide into n_buckets
        quantiles = np.linspace(0, 1, n_buckets + 1)
        bounds = group['ster_grp'].quantile(quantiles).values
        # Round bounds to the nearest multiple of `rounding_base`
        rounded_bounds = np.round(bounds / rounding_base) * rounding_base
        rounded_bounds[-1] = np.inf  # Ensure the last bound is infinity
        bucket_bounds[tg] = rounded_bounds
    return bucket_bounds

def filter_outliers(df):
    q_low = df['ster_grp'].quantile(0.01)
    q_hi  = df['ster_grp'].quantile(0.97)
    df = df[(df['ster_grp'] < q_hi) & (df['ster_grp'] > q_low)]

    q_low = df['ster_1+'].quantile(0.01)
    q_hi  = df['ster_1+'].quantile(0.97)
    df = df[(df['ster_1+'] < q_hi) & (df['ster_1+'] > q_low)]

    return df

def assign_buckets_dynamic(row, bounds_dict):
    bounds = bounds_dict.get(row['target_group'], [0, np.inf])  # Default if no bounds
    for i in range(len(bounds) - 1):
        if bounds[i] <= row['ster_grp'] < bounds[i + 1]:
            return i
    return len(bounds) - 2  # Fallback for any unexpected cases

In [22]:
query = construct_query()
df = query_to_df(query)

In [ ]:
df = filter_outliers(df)
bucket_bounds = calculate_rounded_bucket_bounds(df, n_buckets=4, rounding_base=10)

# Assign buckets dynamically
df['bucket'] = df.apply(lambda row: assign_buckets_dynamic(row, bucket_bounds), axis=1)

df.head(2)


,flight_id,target_group,start_date,end_date,ster_grp,ster_1+,bucket
0,3de5dbf8-e2b5-2c2c-1d59-7b22855dd337,M20-49,2018-01-01,2018-01-12,209.91,47.40,3
1,3de5dbf8-e2b5-2c2c-1d59-7b22855dd337,V25-59,2018-01-01,2018-01-12,353.28,61.26,3


In [32]:
# Separate pre- and post-08-2023 data
df['start_date'] = pd.to_datetime(df['start_date'])
pre_data = df[df['start_date'] < '2023-08-01']
post_data = df[df['start_date'] >= '2023-08-01']

# Calculate correction factors for each target group and bucket
correction_factors = {}
for (tg, bucket), group in pre_data.groupby(['target_group', 'bucket']):
    # Average `ster_1+` for pre-08-2023
    pre_avg = group['ster_1+'].mean()

    # Average `ster_1+` for post-08-2023
    post_avg = post_data[
        (post_data['target_group'] == tg) & (post_data['bucket'] == bucket)
    ]['ster_1+'].mean()

    # Calculate correction factor and round to 2 decimals
    if pd.notna(pre_avg) and pd.notna(post_avg) and pre_avg != 0:
        correction_factors[(tg, bucket)] = round(post_avg / pre_avg, 2)
    else:
        correction_factors[(tg, bucket)] = None  # Handle cases with no data

# Final correction factors dictionary
correction_factors

{('13+', 0): 1.02,
 ('13+', 1): 0.93,
 ('13+', 2): 0.91,
 ('13+', 3): 0.92,
 ('20-49', 0): 0.96,
 ('20-49', 1): 0.94,
 ('20-49', 2): 0.91,
 ('20-49', 3): 0.94,
 ('25+', 0): 1.02,
 ('25+', 1): 0.93,
 ('25+', 2): 0.91,
 ('25+', 3): 0.93,
 ('25-39', 0): 0.94,
 ('25-39', 1): 0.96,
 ('25-39', 2): 0.94,
 ('25-39', 3): 0.96,
 ('25-54', 0): 0.95,
 ('25-54', 1): 0.92,
 ('25-54', 2): 0.91,
 ('25-54', 3): 0.93,
 ('25-59', 0): 0.96,
 ('25-59', 1): 0.93,
 ('25-59', 2): 0.89,
 ('25-59', 3): 0.92,
 ('25-59AB1', 0): 0.97,
 ('25-59AB1', 1): 0.93,
 ('25-59AB1', 2): 0.91,
 ('25-59AB1', 3): 0.93,
 ('25-67', 0): 0.97,
 ('25-67', 1): 0.92,
 ('25-67', 2): 0.89,
 ('25-67', 3): 0.92,
 ('25-67AB1', 0): 1.0,
 ('25-67AB1', 1): 0.94,
 ('25-67AB1', 2): 0.91,
 ('25-67AB1', 3): 0.93,
 ('3-12', 0): 0.98,
 ('3-12', 2): 0.97,
 ('3-12', 3): 0.98,
 ('30-65', 0): 0.96,
 ('30-65', 1): 0.91,
 ('30-65', 2): 0.89,
 ('30-65', 3): 0.92,
 ('35+', 0): 1.04,
 ('35+', 1): 0.93,
 ('35+', 2): 0.91,
 ('35+', 3): 0.94,
 ('35-59', 0): 0.

In [33]:
bucket_bounds


{'13+': array([  0.,  30.,  70., 150.,  inf]),
 '20-49': array([ 0., 10., 30., 70., inf]),
 '25+': array([  0.,  40.,  80., 160.,  inf]),
 '25-39': array([ 0., 10., 20., 50., inf]),
 '25-54': array([ 0., 20., 40., 80., inf]),
 '25-59': array([  0.,  20.,  40., 100.,  inf]),
 '25-59AB1': array([ 0., 20., 40., 80., inf]),
 '25-67': array([  0.,  30.,  60., 120.,  inf]),
 '25-67AB1': array([  0.,  20.,  50., 100.,  inf]),
 '3-12': array([ 0., 10., 10., 30., inf]),
 '30-65': array([  0.,  30.,  60., 120.,  inf]),
 '35+': array([  0.,  40.,  90., 170.,  inf]),
 '35-59': array([  0.,  20.,  50., 110.,  inf]),
 '35-64': array([  0.,  30.,  60., 130.,  inf]),
 '50+': array([  0.,  40.,  90., 170.,  inf]),
 '6+': array([  0.,  30.,  70., 140.,  inf]),
 '68+': array([  0.,  40., 100., 170.,  inf]),
 '7-12': array([ 0., 10., 10., 30., inf]),
 'B2B': array([ 0., 20., 40., 90., inf]),
 'BDS+KND': array([ 0., 20., 40., 80., inf]),
 'BDS-KZ12': array([ 0., 20., 30., 70., inf]),
 'BDS25-59': array([  